In [1]:
from bs4 import BeautifulSoup
import urllib.request
import pandas as pd
import datetime
import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
from collections import defaultdict
import random

In [2]:
def login():   
    my_id = 'YOUR_ID'
    my_pw = 'YOUR_PASSWORD'
    #전체화면용 옵션
    options = webdriver.ChromeOptions()
    #전체화면으로 키기
    options.add_argument("--start-maximized")
    wd = webdriver.Chrome(options=options)
    wd.get("https://thevc.kr/")
    time.sleep(random.randint(2,5))
    login_button = wd.find_element(By.CSS_SELECTOR, 'button.box-button.main.medium')
    login_button.click()
    time.sleep(2)
    id_box = wd.find_element(By.ID,'email')
    id_box.send_keys(my_id)
    pw_box = wd.find_element(By.ID,'password')
    pw_box.send_keys(my_pw)
    time.sleep(1)
    log_button = wd.find_element(By.CSS_SELECTOR,'button.box-button.main.large.mt-40.w-full')
    log_button.click()
    html = wd.page_source
    get_t=BeautifulSoup(html,'html.parser')
    time.sleep(2)
    return wd

In [3]:
#주간 월간 일간
def next_dayl(wd,i):
    lank_button = wd.find_element(By.CSS_SELECTOR, 'h2.text-20 .dropdown-wrap button.text-button')
    wd.execute_script("arguments[0].scrollIntoView({block: 'center'});", lank_button)
    time.sleep(1)
    wd.execute_script("arguments[0].click();", lank_button)
    time.sleep(1)
    buttons_l = wd.find_elements(By.CSS_SELECTOR, 'h2.text-20 .list-20 button')
    buttons_l[i].click()
def next_daym(wd,i):
    element = wd.find_element(By.XPATH, f"//h2[contains(., '{"투자/M&A 랭킹"}')]")
    mn_button = element.find_element(By.CSS_SELECTOR, 'button.text-button')
    wd.execute_script("arguments[0].scrollIntoView({block: 'center'});", mn_button)
    time.sleep(1)
    wd.execute_script("arguments[0].click();", mn_button)
    time.sleep(1)
    buttons_m = element.find_elements(By.CSS_SELECTOR, '.list-20 button')
    buttons_m[i].click()


In [4]:
#다음 카테고리 클릭하고 실행
def next_tabl(wd,lank_tabs_l,current_index_l):
    if current_index_l < len(lank_tabs_l):
        target_tab = lank_tabs_l[current_index_l]
        cat_name = target_tab.text
        wd.execute_script("arguments[0].click();", target_tab)
        time.sleep(1)
        html = wd.page_source
        get_t=BeautifulSoup(html,'html.parser')
        return get_t ,cat_name
    else:
        return None, None
    # # start_tab = wd.find_element(By.CSS_SELECTOR,'div.tab-content')
    # # start_tab.click()
    # time.sleep(3)
    

In [5]:
#다음 카테고리 클릭하고 실행
def next_tabm(wd,lank_tabs_m,current_index_m):
    if current_index_m < len(lank_tabs_m):
        target_tab = lank_tabs_m[current_index_m]
        cat_name = target_tab.text
        wd.execute_script("arguments[0].click();", target_tab)
        time.sleep(1)
        html = wd.page_source
        get_t=BeautifulSoup(html,'html.parser')
        return get_t ,cat_name
    else:
        return None, None
    # # start_tab = wd.find_element(By.CSS_SELECTOR,'div.tab-content')
    # # start_tab.click()
    # time.sleep(3)
    

In [6]:
def con_csv(all_data, j, catagory):

    flat_data = []
    for tab in all_data:
        tab_name = tab['tab_name']
        for rank_item in tab['rank']:
            # rank_item은 {'기업명': '제품명'} 형태입니다.
            for company, product in rank_item.items():
                flat_data.append({
                    "탭이름": tab_name,
                    "기업명": company,
                    "제품명": product
                })

    df = pd.DataFrame(flat_data)

    df.to_csv(f"{catagory}_{j}.csv", index=False, encoding='cp949')

    print("CSV저장")

In [7]:
def rank_c(wd):
    for j in range(3):
        next_dayl(wd,j)
        #카테고리 위치
        tabs = wd.find_elements(By.CSS_SELECTOR, 'div.tab-content')
        lank_tabs = tabs[15:26]
        # print(tabs)
        # for i, tab in enumerate(tabs):
        #     print(f"{i}번째 탭 이름: {tab.text}")
        #랭킹 별로 저장하기
        all_data = []
        for i in range(len(lank_tabs)):
            idx = i
            time.sleep(2)
            result = next_tabl(wd,lank_tabs,idx)
            get_t, cat_name = result
            items = get_t.select('ol.list-wrap > li')
        # print(items)
            now_tab = []
            for item in items[:10]:
                link_tag = item.select_one('a')
                name = link_tag.select_one('div.text-truncate.text-16')
                company = name.contents[0].strip()
                product = name.select_one('div.text-truncate')
                    # print(company)
                    # print(product.text)
                now_tab.append({company:product.text})
            all_data.append({"tab_name":cat_name,
                            "rank": now_tab})    
        print(all_data)
        con_csv(all_data, j,"rank_data")
    return None

In [8]:
def ma_c(wd):    
    for j in range(3):
        next_daym(wd,j)
        #카테고리 위치
        tabs = wd.find_elements(By.CSS_SELECTOR, 'div.tab-content')
        lank_tabs_m = tabs[26:28]
        # print(tabs)
        # for i, tab in enumerate(tabs):
        #     print(f"{i}번째 탭 이름: {tab.text}")
        #랭킹 별로 저장하기
        all_data = []
        for i in range(len(lank_tabs_m)):
            idx = i
            time.sleep(2)
            result = next_tabm(wd,lank_tabs_m,idx)
            get_t, cat_name = result
            items = get_t.select('ol.list-wrap > li')
        # print(items)
            now_tab = []
            for item in items[10:20]:
                rank_d = {}
                link_tag = item.select_one('a')
                if not link_tag:
                    continue
                name = link_tag.select_one('div.text-truncate.text-16')
                if name:
                    company = name.contents[0].strip()
                    product_tag = name.select_one('div.text-truncate')
                    if product_tag:    
                        product_text = product_tag.text.strip()
                    else:
                        product_text = "정보없음"
                    # print(company)
                    # print(product.text)
                    now_tab.append({company:product_text})
                else:
                    continue
            all_data.append({"tab_name":cat_name,
                            "rank": now_tab})
        print(all_data)
    con_csv(all_data,j,"d&m_data")
    return None

In [9]:
def main():
    wd = login()
    html = wd.page_source
    get_t=BeautifulSoup(html,'html.parser')
    #카테고리 위치
    tabs = wd.find_elements(By.CSS_SELECTOR, 'div.tab-content')

    rank_c(wd)
    ma_c(wd)


if __name__ == "__main__":
    main()


[{'tab_name': '전체', 'rank': [{'리얼월드': '피지컬 AI 모델'}, {'알세미': '알시스'}, {'포인투테크놀로지': 'E-tube'}, {'에이로봇': '앨리스'}, {'네이버클라우드': '네이버 클라우드 플랫폼'}, {'모레': 'MoAI 플랫폼'}, {'앤트로픽': '클로드'}, {'퓨리오사에이아이': 'AI 반도체'}, {'원지랩스': '마이타로'}, {'모드하우스': '연예매니지먼트'}]}, {'tab_name': '스타트업', 'rank': [{'리얼월드': '피지컬 AI 모델'}, {'알세미': '알시스'}, {'포인투테크놀로지': 'E-tube'}, {'에이로봇': '앨리스'}, {'모레': 'MoAI 플랫폼'}, {'앤트로픽': '클로드'}, {'퓨리오사에이아이': 'AI 반도체'}, {'원지랩스': '마이타로'}, {'모드하우스': '연예매니지먼트'}, {'리벨리온': 'AI 반도체'}]}, {'tab_name': '액셀러레이터', 'rank': [{'리얼비즌': '액셀러레이팅'}, {'김기사랩': '김기사랩'}, {'연세대학교기술지주': '기술지주회사'}, {'스파크랩': '액셀러레이팅'}, {'네스트컴퍼니': '액셀러레이팅'}, {'한국투자액셀러레이터': '액셀러레이팅'}, {'메디이노파트너스': '액셀러레이팅'}, {'하임벤처투자': '액셀러레이팅'}, {'콴티파이인큐베이터': '액셀러레이팅'}, {'해시드': '액셀러레이팅'}]}, {'tab_name': '벤처캐피탈', 'rank': [{'엘지테크놀로지벤처스': '투자조합'}, {'베이스벤처스': '투자조합'}, {'보광인베스트먼트': '투자조합'}, {'에이스톤벤처스': '투자조합'}, {'삼성넥스트': '투자조합'}, {'케이비인베스트먼트': '투자조합'}, {'엘비인베스트먼트': '투자조합'}, {'에이지인베스트먼트': '투자조합'}, {'가이아벤처파트너스': '투자조합'}, {'무신사파트너스': '투자조합'}]}, {'tab_name': '사모투자회사

In [10]:
# def main():
#     wd = login()
#     html = wd.page_source
#     get_t=BeautifulSoup(html,'html.parser')
#     #카테고리 위치
#     tabs = wd.find_elements(By.CSS_SELECTOR, 'div.tab-content')
#     lank_tabs_l = tabs[14:25]
#     current_index_l = 0
#     lank_tabs_m = tabs[26:28]
#     current_index_m = 0

#     for j in range(3):
#         next_dayl(j)
#         #카테고리 위치
#         tabs = wd.find_elements(By.CSS_SELECTOR, 'div.tab-content')
#         lank_tabs = tabs[14:25]
#         current_index_l = 0
#         # print(tabs)
#         # for i, tab in enumerate(tabs):
#         #     print(f"{i}번째 탭 이름: {tab.text}")
#         #랭킹 별로 저장하기
#         all_data = []
#         for i in range(len(lank_tabs)):
#             time.sleep(2)
#             result = next_tabl()
#             get_t, cat_name = result
#             items = get_t.select('ol.list-wrap > li')
#         # print(items)
#             now_tab = []
#             num = 0
#             for item in items[:10]:
#                 rank_d = {}
#                 link_tag = item.select_one('a')
#                 name = link_tag.select_one('div.text-truncate.text-16')
#                 company = name.contents[0].strip()
#                 product = name.select_one('div.text-truncate')
#                     # print(company)
#                     # print(product.text)
#                 now_tab.append({company:product.text})
#             all_data.append({"tab_name":cat_name,
#                             "rank": now_tab})    
#         print(all_data)

#         flat_data = []
#         for tab in all_data:
#             tab_name = tab['tab_name']
#             for rank_item in tab['rank']:
#                 # rank_item은 {'기업명': '제품명'} 형태입니다.
#                 for company, product in rank_item.items():
#                     flat_data.append({
#                         "탭이름": tab_name,
#                         "기업명": company,
#                         "제품명": product
#                     })

#         df = pd.DataFrame(flat_data)

#         df.to_csv(f"startup_rankings_{j}.csv", index=False, encoding='cp949')

#         print("CSV 저장 완료!")


#     for j in range(3):
#         next_daym(j)
#         #카테고리 위치
#         tabs = wd.find_elements(By.CSS_SELECTOR, 'div.tab-content')
#         lank_tabs_m = tabs[26:28]
#         current_index_m = 0
#         # print(tabs)
#         # for i, tab in enumerate(tabs):
#         #     print(f"{i}번째 탭 이름: {tab.text}")
#         #랭킹 별로 저장하기
#         all_data = []
#         for i in range(len(lank_tabs_m)):
#             time.sleep(2)
#             result = next_tabm()
#             get_t, cat_name = result
#             items = get_t.select('ol.list-wrap > li')
#         # print(items)
#             now_tab = []
#             num = 0
#             for item in items[10:20]:
#                 rank_d = {}
#                 link_tag = item.select_one('a')
#                 if not link_tag:
#                     continue
#                 name = link_tag.select_one('div.text-truncate.text-16')
#                 if name:
#                     company = name.contents[0].strip()
#                     product_tag = name.select_one('div.text-truncate')
#                     if product_tag:    
#                         product_text = product_tag.text.strip()
#                     else:
#                         product_text = "정보없음"
#                     # print(company)
#                     # print(product.text)
#                     now_tab.append({company:product_text})
#                 else:
#                     continue
#             all_data.append({"tab_name":cat_name,
#                             "rank": now_tab})    
#         print(all_data)   

#         flat_data = []
#         for tab in all_data:
#             tab_name = tab['tab_name']
#             for rank_item in tab['rank']:
#                 for company, product in rank_item.items():
#                     flat_data.append({
#                         "탭이름": tab_name,
#                         "기업명": company,
#                         "제품명": product
#                     })

#         df = pd.DataFrame(flat_data)

#         df.to_csv(f"m&a_rankings_{j}.csv", index=False, encoding='cp949')

#         print("CSV 저장 완료!")